In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "nyctaxi_project"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"
LANDING_PATH = f"{BASE}/trips"
CHECKPOINT_PATH = f"{BASE}/_checkpoints/bronze_trips"
SCHEMA_PATH = f"{BASE}/_schemas/bronze_trips"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_trips"


In [0]:
bronze_stream = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", True)
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes", False) # Bronze castea todo como string.
        .option("pathGlobFilter", "*.csv")            # Ignora _SUCCESS, _committed, etc.
        .load(LANDING_PATH)
        .select(
            "*",
            F.col("_metadata.file_path").alias("source_files"),
            F.current_timestamp().alias("ingestion_ts"),
        )
)

In [0]:
query = (
    bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True) # procesa lo pendiente y termina (compatible con serverless).
    .toTable(BRONZE_TABLE)
)
query.awaitTermination()
print("Bronze cargada")

Bronze cargada


In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS filas_bronze,
           COUNT(DISTINCT source_files) AS archivos_leidos
    FROM {BRONZE_TABLE}
""").show()

display(spark.table(BRONZE_TABLE).limit(5))

+------------+---------------+
|filas_bronze|archivos_leidos|
+------------+---------------+
|       21932|             60|
+------------+---------------+



tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,_rescued_data,source_files,ingestion_ts
2016-02-11T18:39:25.000Z,2016-02-11T18:44:26.000Z,0.71,5.5,10021,10065,null,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z
2016-02-11T08:13:16.000Z,2016-02-11T08:18:27.000Z,0.7,5.5,10003,10012,null,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z
2016-02-11T15:26:49.000Z,2016-02-11T15:36:21.000Z,1.0,7.5,10028,10044,null,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z
2016-02-11T08:21:31.000Z,2016-02-11T08:33:31.000Z,2.16,10.0,10119,10012,null,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z
2016-02-11T22:42:46.000Z,2016-02-11T23:00:30.000Z,2.4,13.0,10022,10003,null,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z


In [0]:
dias_cargados = [r["d"] for r in spark.sql(f"""
    SELECT DISTINCT regexp_extract(source_files, '/trips/([0-9-]{{10}})/', 1) AS d
    FROM {BRONZE_TABLE}
""").collect()]

filas_origen = (
    spark.table("samples.nyctaxi.trips")
    .filter(F.to_date("tpep_pickup_datetime").cast("string").isin(dias_cargados))
    .count()
)
print("Filas en origen para esos días:", filas_origen)

Filas en origen para esos días: 21932
